# Hard Thresholding refined search on Fritz

This notebook refines the rank grid around the coarse-search winner and uses longer trajectories to select the retained iteration.

In [ ]:
import sys
from pathlib import Path
from time import perf_counter

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.sparse.linalg import ArpackNoConvergence
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from hard_thresholding_hpc import (
    hard_thresholding_model_matrix_free,
    predict_pairs,
)

In [ ]:
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed" / "ml-latest"
COARSE_RESULTS_DIR = PROJECT_ROOT / "results" / "hard_thresholding_coarse"
RESULTS_DIR = PROJECT_ROOT / "results" / "hard_thresholding_refined"
HISTORIES_DIR = RESULTS_DIR / "histories"
HISTORIES_DIR.mkdir(parents=True, exist_ok=True)

train_df = pd.read_csv(PROCESSED_DIR / "train_ratings.csv")
user_map = pd.read_csv(PROCESSED_DIR / "user_map.csv")
movie_map = pd.read_csv(PROCESSED_DIR / "movie_map.csv")

n_users = len(user_map)
n_movies = len(movie_map)

train_fit_df, val_df = train_test_split(
    train_df,
    test_size=0.2,
    random_state=42,
    shuffle=True,
)

val_user_idx = val_df["user_idx"].to_numpy(dtype=np.int64)
val_movie_idx = val_df["movie_idx"].to_numpy(dtype=np.int64)
val_ratings = val_df["rating"].to_numpy(dtype=np.float64)

print("Train fit shape:", train_fit_df.shape)
print("Validation shape:", val_df.shape)
print("Number of users:", n_users)
print("Number of movies:", n_movies)

In [ ]:
coarse_results_path = COARSE_RESULTS_DIR / "best_by_rank.csv"
if not coarse_results_path.exists():
    raise FileNotFoundError(
        "Run the coarse-search Slurm job before the refined search: "
        f"{coarse_results_path}"
    )

coarse_best_by_rank = pd.read_csv(coarse_results_path)
print("Best coarse-search results:")
display(coarse_best_by_rank)

if (coarse_best_by_rank["iteration"] == 15).all():
    print("Every coarse trajectory was best at iteration 15; longer runs are required.")

In [ ]:
def hard_validation_metrics(U, singular_values, V):
    predictions = predict_pairs(
        U,
        singular_values,
        V,
        val_user_idx,
        val_movie_idx,
    )
    predictions = np.clip(predictions, 0.5, 5.0)
    return {
        "val_rmse": float(np.sqrt(np.mean((val_ratings - predictions) ** 2))),
    }

## Refined rank grid

The coarse optimum was rank 32. Ranks 20-48 refine its neighborhood; rank 64 is not repeated because its validation RMSE was already clearly worse.

In [ ]:
refined_rank_values = [20, 24, 28, 32, 36, 40, 48]
refined_grid_max_iter = 60
finalist_max_iter = 120
hard_svd_tol = 1e-4
hard_svd_maxiter = 300

print("Refined ranks:", refined_rank_values)
print("Iterations per refined rank:", refined_grid_max_iter)
print("Finalist iterations:", finalist_max_iter)

## Refined search

In [ ]:
# Compile the Numba prediction kernel before the timed search.
predict_pairs(
    np.ones((2, 1)),
    np.ones(1),
    np.ones((2, 1)),
    np.array([0], dtype=np.int64),
    np.array([0], dtype=np.int64),
)

refined_histories = []
refined_failed_configs = []
refined_search_start = perf_counter()

for config_number, rank in enumerate(refined_rank_values, start=1):
    history_path = HISTORIES_DIR / f"rank-{rank}.csv"
    print(f"[{config_number}/{len(refined_rank_values)}] rank={rank}")

    if history_path.exists():
        print("  using saved history")
        refined_histories.append(pd.read_csv(history_path))
        continue

    config_start = perf_counter()
    try:
        _, _, _, history_df = hard_thresholding_model_matrix_free(
            train_fit_df,
            n_users=n_users,
            n_movies=n_movies,
            rank=rank,
            n_iter=refined_grid_max_iter,
            metrics_fn=hard_validation_metrics,
            svd_tol=hard_svd_tol,
            svd_maxiter=hard_svd_maxiter,
            random_state=42,
        )
    except ArpackNoConvergence as error:
        refined_failed_configs.append({
            "rank": rank,
            "error": repr(error),
        })
        print("  failed to converge:", repr(error))
        continue

    config_seconds = perf_counter() - config_start
    history_df["config_seconds"] = config_seconds
    history_df.to_csv(history_path, index=False)
    refined_histories.append(history_df)
    print(
        f"  {config_seconds / 60:.1f} min; "
        f"best val RMSE={history_df['val_rmse'].min():.6f}"
    )

if not refined_histories:
    raise RuntimeError("Every refined hard-thresholding configuration failed")

refined_results_df = pd.concat(refined_histories, ignore_index=True)
refined_best_by_rank = (
    refined_results_df.loc[
        refined_results_df.groupby("rank")["val_rmse"].idxmin()
    ]
    .sort_values("val_rmse")
    .reset_index(drop=True)
)
refined_results_df.to_csv(RESULTS_DIR / "all_histories.csv", index=False)
refined_best_by_rank.to_csv(RESULTS_DIR / "best_by_rank.csv", index=False)

if refined_failed_configs:
    pd.DataFrame(refined_failed_configs).to_csv(
        RESULTS_DIR / "failed_configs.csv",
        index=False,
    )

print(f"Refined search time: {(perf_counter() - refined_search_start) / 60:.1f} min")
display(refined_best_by_rank)

## Extend the best rank and select the retained iteration

In [ ]:
best_refined_row = refined_best_by_rank.iloc[0]
best_hard_rank = int(best_refined_row["rank"])

print(f"Extending rank={best_hard_rank} to {finalist_max_iter} iterations...")
_, _, _, finalist_history = hard_thresholding_model_matrix_free(
    train_fit_df,
    n_users=n_users,
    n_movies=n_movies,
    rank=best_hard_rank,
    n_iter=finalist_max_iter,
    metrics_fn=hard_validation_metrics,
    svd_tol=hard_svd_tol,
    svd_maxiter=hard_svd_maxiter,
    random_state=42,
)

best_iteration_row = finalist_history.loc[
    finalist_history["val_rmse"].idxmin()
]
best_hard_n_iter = int(best_iteration_row["iteration"])
best_hard_val_rmse = float(best_iteration_row["val_rmse"])

best_parameters_df = pd.DataFrame([{
    "rank": best_hard_rank,
    "n_iter": best_hard_n_iter,
    "val_rmse": best_hard_val_rmse,
}])
finalist_history.to_csv(RESULTS_DIR / "finalist_history.csv", index=False)
best_parameters_df.to_csv(RESULTS_DIR / "best_parameters.csv", index=False)
display(best_parameters_df)

if best_hard_n_iter == finalist_max_iter:
    print("Warning: validation RMSE is still best at the final iteration; extend the trajectory again.")

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(finalist_history["iteration"], finalist_history["train_rmse"], label="Train RMSE")
plt.plot(finalist_history["iteration"], finalist_history["val_rmse"], label="Validation RMSE")
plt.axvline(
    best_hard_n_iter,
    linestyle="--",
    color="black",
    label=f"Retained iteration = {best_hard_n_iter}",
)
plt.xlabel("Iteration")
plt.ylabel("RMSE")
plt.title(f"Hard Thresholding: train vs validation RMSE (rank={best_hard_rank})")
plt.legend()
plt.grid(alpha=0.3)
plt.show()